<a href="https://colab.research.google.com/github/zeeldhorajiya1909/prodigy_tasks/blob/main/task1_prodigy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade transformers[torch] datasets accelerate

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset

# 1. Load Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 needs a padding token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

# 2. Load and Preprocess Dataset
dataset = load_dataset("rajpurkar/squad", split="train[:1%]")

def tokenize_function(examples):
    return tokenizer(examples["context"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

# 3. Data Collator for Language Modeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 4. Set up Training Arguments
training_args = TrainingArguments(
    output_dir="./gpt2-fine-tuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=100,
    save_total_limit=2,
    logging_steps=10,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

# 5. Initialize Trainer and Start Fine-tuning
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

print("Starting fine-tuning...")
trainer.train()

# 6. Save the model
trainer.save_model("./gpt2-fine-tuned")
tokenizer.save_pretrained("./gpt2-fine-tuned")
print("Model saved successfully!")

In [ ]:
from transformers import pipeline, GenerationConfig

# Load the fine-tuned model
model_path = "./gpt2-fine-tuned"
generator = pipeline("text-generation", model=model_path, tokenizer=model_path)

# Create a generation configuration to handle parameters cleanly
config = GenerationConfig(
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.8,
    max_new_tokens=100,
    pad_token_id=generator.tokenizer.eos_token_id,
    clean_up_tokenization_spaces=False
)

# Test the model using the config
prompt = "the quick brown fox jump over the lazy dog."
results = generator(prompt, generation_config=config)

print("Generated Text:")
print(results[0]['generated_text'])